# Agente de revisión de documentos y planos (API de Claude, de pago)
Requiere *Internet: On* y el secreto `ANTHROPIC_API_KEY` en *Add-ons → Secrets*. Mayor calidad: lee planos con visión y PDF con tablas. Sube también `plan_review.py` y `checklist_planos.md`.

In [ ]:
!pip install -q anthropic pypdf python-docx openpyxl pymupdf pillow

In [ ]:
import os, sys, glob
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
hits = glob.glob("/kaggle/input/**/doc_review_agent.py", recursive=True) + glob.glob("/kaggle/working/doc_review_agent.py")
sys.path.insert(0, os.path.dirname(hits[0]))
from doc_review_agent import ReviewConfig, review_folder
resultado = review_folder("/kaggle/input/documentos-ingenieria", "/kaggle/working/informe", ReviewConfig())

## Chat con los documentos
Haz preguntas, pide resúmenes o consulta los hallazgos de la revisión. Responde solo con lo que encuentra en los documentos y cita archivo y página.

## Interpretación de planos
Extrae de cada hoja qué muestra: tipo de plano, cajetín, elementos, cotas, materiales, notas y referencias. Se guarda en `planos_interpretados.md/json` y alimenta el chat (si tu carpeta no tiene planos, omite esta celda y la línea `add_plans`).

In [ ]:
from plan_review import interpret_folder
planos = interpret_folder(IN_DIR, "/kaggle/working/informe", cfg, llm=llm)
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/informe/planos_interpretados.md", encoding="utf-8").read()))

In [ ]:
from doc_review_agent import ClaudeLLM
llm = ClaudeLLM(ReviewConfig())
from doc_chat import DocChat
chat = DocChat(llm)
IN_DIR = "/kaggle/input/documentos-ingenieria"
chat.add_folder(IN_DIR)
chat.add_review("/kaggle/working/informe/revision.json")
chat.add_plans(planos)   # planos interpretados: el chat los usa y mira la hoja relevante

In [ ]:
print(chat.ask("¿Cuáles son los hallazgos críticos y mayores?"))
print(chat.ask("¿Qué norma se aplica y qué factores de seguridad se usan?"))
print(chat.ask("¿Qué muestra el plano de la hoja 1 y cuáles son sus cotas principales?"))
print(chat.ask("¿Qué materiales y normas se indican en los planos?"))

In [ ]:
print(chat.summarize())            # todos los documentos; o chat.summarize("archivo.pdf")

In [ ]:
print(chat.ask_image("¿Las cotas parciales suman la cota total? ¿Qué datos faltan en el cajetín?",
                     "/kaggle/input/documentos-ingenieria/plano.pdf", page=1))

In [ ]:
chat.repl()   # chat interactivo: escribe tus preguntas; /resumen, /reset, /salir